# Tutorial 13: Generation decoding and the KV cache

**Prerequisite:** lecture 13. **Compute:** CPU; standard library and NumPy.

Predict the result before each code cell. Run cells in order from a fresh kernel. The course bootstrap needs the full extracted course pack. Outputs are intentionally empty; keep your actual measurements.


In [ ]:
from pathlib import Path
import sys,json,time,random,hashlib,math
if hasattr(sys.stdout,'reconfigure'): sys.stdout.reconfigure(encoding='utf-8')
candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'llm_course',Path.cwd()/'llm_course_v2',Path('/content/llm_course'),Path('/content/llm_course_v2')]
COURSE=next((p for p in candidates if (p/'coursekit/core.py').exists()),None)
if COURSE is None: raise RuntimeError('Extract the complete course pack; run from its root or notebooks directory.')
if str(COURSE) not in sys.path:sys.path.insert(0,str(COURSE))
RUNS=COURSE/'runs';RUNS.mkdir(exist_ok=True)
from coursekit.core import *
random.seed(7)
print('Course root:',COURSE)


First perform greedy versus seeded sampling on a three-word vocabulary. This isolates decoding from training. The optional next cell uses a real pretrained model to complete an ordinary school sentence; it introduces no retrieval system.


In [ ]:
import numpy as np
p=np.array([.6,.2,.2]);words=np.array(['bus','train','bicycle'])
rng=np.random.default_rng(7)
print('Greedy:',words[p.argmax()]);print('Twenty samples:',rng.choice(words,20,p=p))
for temperature in [.5,1.,2.]:
    z=np.log(p)/temperature;e=np.exp(z-z.max());print(temperature,e/e.sum())


### Pause before the next step
Explain why the most likely first token need not lead to the most likely complete sequence.


In [ ]:
RUN_MODEL=False
if RUN_MODEL:
    import subprocess
    subprocess.check_call([sys.executable,'-m','pip','install','transformers>=4.45,<5','torch>=2.4,<3'])
    import torch
    from transformers import AutoTokenizer,AutoModelForCausalLM
    model_id='Qwen/Qwen2.5-0.5B'
    tokenizer=AutoTokenizer.from_pretrained(model_id)
    model=AutoModelForCausalLM.from_pretrained(model_id).eval()
    tokens=tokenizer('I go to school by',return_tensors='pt')
    with torch.no_grad():out=model.generate(**tokens,max_new_tokens=20,do_sample=False,pad_token_id=tokenizer.eos_token_id)
    print('Revision',model.config._commit_hash,'IDs',tokens['input_ids'].tolist())
    print(tokenizer.decode(out[0],skip_special_tokens=True))
else:print('Real-model generation skipped. Set RUN_MODEL=True to download and run.')


### Independent experiment
For a real-model run, compare greedy and two seeded samples under the same prefix and maximum length. Record model revision, actual tokens and latency. Explain how a KV cache changes computation but not the intended conditional distribution.

Submit a short report containing your prediction, the actual result, a comparison and a limitation. Record any environment or data changes. Do not substitute an illustrative number for a measured result.
